In [ ]:
!pip install -q yt-dlp

In [ ]:
! pip install pytube openai-whisper openai

In [ ]:
!pip install pytubefix

In [ ]:
from pytubefix import YouTube

In [ ]:
!pip install git+https://github.com/openai/whisper.git

In [ ]:
!npm install -g youtube-po-token-generator

In [ ]:
!youtube-po-token-generator

In [ ]:
import whisper
model = whisper.load_model("large")
print(model.config)

In [ ]:
import whisper
from pytubefix import YouTube
import os
import openai
import json
from openai import AzureOpenAI
import yt_dlp

!npm install -g youtube-po-token-generator


def download_audio(youtube_url, output_path="audio.mp3"):
    # The output filename needs to be without the extension for yt-dlp
    output_filename = os.path.splitext(output_path)[0]

    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': output_filename,
        # 'cookiefile': 'cookies.txt',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
    }

    print(f"Downloading audio from {youtube_url} with yt-dlp...")
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([youtube_url])

    # The output file will be audio.mp3, so we just return the original intended path
    return output_path



def transcribe_audio(audio_path, model_size="large"):
    model = whisper.load_model(model_size)
    result = model.transcribe(audio_path, task="transcribe", fp16=False)
    return result['text'], result['language']
from openai import AzureOpenAI

def translate_text(text, source_lang):
    client = AzureOpenAI(
        api_key="API_KEY",
        api_version="2023-05-15",
        azure_endpoint="OPEN_AI_ENDPOINT",
        organization="101552",
    )

    prompt = f"""
You are a precise and culturally aware translator.

Translate the full transcript below from {source_lang} to English, ensuring that nothing is omitted.
	•	Preserve the tone, flow, word choice, and cultural nuances as closely as possible.
	•	Do not summarize or paraphrase — your translation must remain faithful to the original structure and style of the speaker.
	•	If any part is already in English, include it as-is in the final output.
"""


    response = client.chat.completions.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": text},
        ],
        temperature=0.7,
        max_tokens=4096,
    )

    return response.choices[0].message.content

def process_video(youtube_url):
    print("Downloading audio...")
    audio_path = download_audio(youtube_url)

    print("Transcribing with Whisper...")
    transcript, language = transcribe_audio(audio_path)
    print(f"Detected language: {language}")

    if language != "en":
        print("Translating to English using GPT...")
        translated = translate_text(transcript, language)
    else:
        translated = transcript

    return transcript, translated

# === USAGE ===
video_url = "YOUTUBE_LINK"
original_text, english_translation = process_video(video_url)

# Save the output
with open("original_transcript.txt", "w", encoding="utf-8") as f:
    f.write(original_text)

with open("translated_transcript.txt", "w", encoding="utf-8") as f:
    f.write(english_translation)

print("Done. Transcripts saved.")

In [ ]:
original_text

In [ ]:
english_translation